# Text-Segmentierung Vision-LLM

**Voraussetzungen**

Environment: dski

Zur Durchführung der Segmentierung müssen die Seiten der PDF-Dokumente in Bilder umgewandelt werden (Preparation.ipynb).

**Versuchsaufbau**

Segmentierung der PDF-Dokumente mittels Vision-LLM.

PDF-Dokumente seitenweise einlesen und Text+Grafik der Seiten über Prompt in LLM eingeben. Rückgabe der Informationen für Segmente in strukturierter Form.

## Setup

### Pakete laden

In [1]:
import base64
import glob
import os
import math
import json
import pandas as pd
import pymupdf
import urllib

from openai import OpenAI

### Variablen+Funktionen

In [2]:
# Pfad zum Datensatz
path = 'vectara_open_ragbench/open_ragbench/pdf/arxiv/'

#apphub.wolke.uni-greifswald.de

# Vision-LLM
model = 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'

# LLM OpenAI-Client
client = OpenAI(
    api_key="",
    base_url="https://apphubai.wolke.uni-greifswald.de/v1"
)

### Daten laden

PDF URLs aus Datensatz laden

In [4]:
pdf_urls = pd.read_json(f'{path}pdf_urls.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: "pdf_urls"})

## Test

### Responses Endpoint

Der Responses-Endpoint stellt eine Weiterentwicklung des Chat-Completion-Endpoints und vereinheitlichte Schnittstelle dar und unterstützt multimodale Eingaben (Text+Bild) sowie mehrstufige Interaktionen. 

Durch apphubai.wolke.uni-greifswald.de wird der Responses-Endpoint nicht unterstützt und kann nicht verwendet werden.

#### File Endpoint

Datei zur Eingabe für die Verarbeitung hinzufügen.

In [ ]:
#file = client.files.create(
#    file=open("24_2-fold_2.png", "rb"),
#    purpose="assistants"
#)

#### Responses request

Responses-Anfrage erstellen und File in Prompt einbeziehen

In [ ]:
#response = client.responses.create(
#    model="Qwen/Qwen3-VL-30B-A3B-Instruct-FP8",
#    input=[
#        {
#            "role": "user",
#            "content": [
#                {"type": "input_text", "text": "Fasse das Dokument zusammen"},
#                {
#                    "type": "input_file",
#                    "file_id": file.id
#                }
#            ]
#        }
#    ]
#)

#print(response)

### Chat-Completition-Endpoint

Standard Chat-Completition-Endpoint

File als Base64-kodierte Zeichenkette laden

#### Chat-Completition-Request

Chat-Completition-Anfrage erstellen und Zeichenkette des Bildes in Prompt einbeziehen.

In [11]:
#f = open(f'{path}/images/2401.01872v2/page-0.png', 'rb')
#encoded_string = base64.b64encode(f.read()).decode("utf-8")
#f.close()

#completion = client.chat.completions.create(
#    model="Qwen/Qwen3-VL-30B-A3B-Instruct-FP8",
#    messages=[
#        {
#            "role": "user",
#            "content": [
#                {"type": "text", "text": "Was ist das wichtigste Transportmittel für Informationen?"},#Unterteile den Text der zweiten Seite des angefügten PDF-Dokuments in inhaltlich zusammenhängende Segmente.
#                {
#                    "type": "image_url",
#                    "image_url": {
#                        #"url": "https://s3-eu-west-1.amazonaws.com/gxmedia.galileo-press.de/pagepreviews/4694/1600/24_2-fold_2.png"
#                        "url": f"data:image/png;base64,{encoded_string}"#application/pdf
#                    }
#                }
#            ]
#        }
#    ],
#)

#print(completion.choices[0].message.content)

## Ausführung

Texte+Bilder der PDF-Dokumente seitenweise über Prompt in LLM eingeben. Über Tool-Calling wird die Funktion "get_text_segments" aufgerufen und die Informationen zu den erkannten Segmenten mit Positionsangaben (left, top) übergeben.

In [32]:
# Über alle Dokumente iterieren
#for doc_id in pdf_urls.index:
for idx_doc in range(0,pdf_urls.shape[0]):
    doc_id = pdf_urls.iloc[idx_doc].name
    # Über alle Bilder der Seiten eines Dokuments iterieren
    #for file in glob.glob(f'{path}images/{doc_id}/*'):

    # PDF mit pymupdf einlesen
    doc = pymupdf.open(f'{path}pdfs/{doc_id}.pdf')

    # Über alle Seiten eines PDF-Dokuments iterieren
    for idx_page in range(0,len(doc)):
        page = doc[idx_page]
        data = {"size":[], "text":[], "bbox":[], "color":[], "page":[], "block_index":[], "line_index":[]}
        blocks = page.get_text("dict")["blocks"]#, flags=11
        # Über alle Bläcke der Seite iterieren
        for idx_block in range(len(blocks)):
            block = blocks[idx_block]
            # Zeilen
            #print(block)
            #print("---")
            if block['type'] == 0:
                # Über alle Zeilen innerhalb eines Blocks iterieren
                #for line in block['lines']:
                for idx_line in range(len(block['lines'])):
                    line = block['lines'][idx_line]
                    #print("line")
                    #print("---")
                    # Über alle Text-Elemente innerhalb einer Zeile iterieren
                    for span in line['spans']:
                        #print(f"span: {span['text']}")
                        #print("---")
                        if len(span["text"].strip()) > 0:# and math.floor(span['size']) != 6:
                            data["size"].append(math.floor(span['size']))
                            data["text"].append(span['text'])
                            data["bbox"].append(block['bbox'])
                            data["color"].append(span['color'])
                            data["page"].append(idx_page)
                            data["line_index"].append(idx_line)
                            data["block_index"].append(idx_block)

        # Pandas Dataframe aus Seitenenelmenten erstellen
        df_page = pd.DataFrame.from_dict(data)
        
        # Datentypen convertieren
        df_page = df_page.convert_dtypes()

        #print(df_page.groupby(["block_index"])["text"].agg(" ".join).reset_index().drop("block_index", axis=1)['text'].agg("\n\n".join))
        
        # Bild der Seite als base64 kodierte Zeichenkette einlesen
        try:
            f = open(f'{path}images/{doc_id}/page-{idx_page}.png', 'rb')
            encoded_string = base64.b64encode(f.read()).decode("utf-8")
            f.close()
        except Exception as e:
            print(f"Exception occurred: {e}")
            continue

        # ChatCompletition erstellen
        completion = client.chat.completions.create(
            model=model,
            temperature=0.0,
            stream=False,
            messages=[
                {
                    'role': 'system',
                    'content': [
                        {
                            'type': 'text',
                            'text': 'User input provides an image representing a page of a document. Segments are to be created based on the information in the image. A new segment is introduced by a change of topic. One or more segments can appear in the page text. The analysis must cover the entire page.\n\nThe JSON object consists of a list in which the top and left coordinates of the bounding box, as well as the first ten characters of the text of each segment, are stored as a dictionary {\'top\':0, \'left\':0, \'text\':\'\'}. Finally, call the function get_text_segments and pass the list with information about the detected segments via the argument segments.'
                        }
                    ]
                },
                {
                    'role': 'user',
                    'content': [
                        {
                            'type': 'text', 
                            'text': 'Perform the segmentation, call the function get_text_segments and pass the information about the detected segments via the argument segments.'
                        },
                        {
                            'type': 'image_url',
                            'image_url': {
                                'url': f'data:image/png;base64,{encoded_string}'#application/pdf
                            }
                        }
                    ]
                }
            ],
            tools=[
                {
                    'type': 'function',
                    'function': {
                        'name': 'get_text_segments', 
                        'description': 'Process information about segments, top and left coordinates.', 
                        'parameters': {
                            'type': 'object', 
                            'properties': {
                                'segments': {
                                    'type': 'array', 
                                    #'items': {
                                    #    'type': 'array',
                                    #    #'enum': ['', 'Undefiniert']
                                    #},
                                    #'minItems': 1, #maxItems
                                    #'uniqueItems': True, 
                                    'description': 'List containing information about segments.',
                                }, 
                            }, 
                            'required': ['segments'],
                        },
                    },
                }
            ]
        )

        # Rückgabe verarbeiten
        print(completion)
        #print(completion.choices[0].message.content)
        if not(completion.choices[0].message.tool_calls):
            print("No tool call", end='\n\n')
        else:
            json_resp = json.loads(completion.choices[0].message.tool_calls[0].function.arguments)
            print(json_resp)
        
        break
    
    break

ChatCompletion(id='chatcmpl-hemQp6NpJoEvTPhYK5OaemPIhtioOeDY', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='0Bsg9DXEVpi31zLjAQujkn8fnqYTKVw1', function=Function(arguments='{"segments":[{"top": 135, "left": 178, "text": "DISCRETE APPROXIMATION OF RISK"}, {"top": 185, "left": 197, "text": "JONAS BLESSING, MICHAEL KUPPER, AND"}, {"top": 220, "left": 207, "text": "ABSTRACT. We discuss the asymptotic behaviour of risk-based"}, {"top": 365, "left": 207, "text": "Key words: risk-based pricing, indifference pricing, volatility"}, {"top": 383, "left": 207, "text": "MSC 2020 Classification: Primary 91G20, 47H20; Secondary 91G70, 91G60, 47J25."}, {"top": 442, "left": 417, "text": "1. INTRODUCTION"}, {"top": 477, "left": 150, "text": "Computing the prices of financial derivatives strongly depends on 